In [1]:
%autoreload

In [2]:
import sys
print(sys.executable)
import keras
print(keras.__version__)

/home/dhkim/research/Machine_Learning/.pixi/envs/default/bin/python


2026-07-22 22:33:38.657518: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


3.12.0


In [3]:
ssptransfer = None
src_model = "KACE-1-0-G"  
tgt_model = "EC-Earth3"  

tfmodel = 'transformer'

home_dir = os.path.expanduser('~/')
data_dir = home_dir+'/data/'
ML_dir = home_dir+'/research/Machine_Learning/MLresults/'
SSPLIST = [126,245,585]

set_input = 10 
selmodes = 8
pclen_loop = 10
year_start = 1995
year_end = 2100

ny, nx = 512, 512
nt = 1272
nt_seq = nt - 3

titleabc = ["A ", "B ", "C ", "D ", "E ", "F ", "G ", "H "]
title_ssp=["SSP1-2.6", "SSP2-4.5","SSP5-8.5"]


xr_ns_to_year = 1e9 * 60 * 60 * 24 * 365.25  


In [4]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import xarray as xr
import matplotlib.pyplot as plt
from glob import glob
import matplotlib.gridspec as gridspec
import matplotlib.lines as mlines
from matplotlib.legend_handler import HandlerTuple



from npj_analysis_func import *

# load dataset

In [5]:
TIME = pd.date_range("1995-01", "2100-12", freq="MS") + pd.offsets.Day(14)
TIME_seq = pd.date_range("1995-04", "2100-12", freq="MS") + pd.offsets.Day(14)
TIME_yearly = pd.date_range("1995-04", "2100-12", freq="YS") + pd.offsets.Day(14) 
year_start = 1995
seq_length = 4
seqnt = seq_length - 1

varY_name = "SST"  

varlist_X = ["tas", "uas", "vas", "huss", "rlds", "rsds"]
var_trendX = ["tas_PC1", "huss_PC1", "rlds_PC1"]
var_trendY = "SST_PC1"
var_Physical = [
    "tas_PC1",
    "uas_PC1",
    "vas_PC1",
    "huss_PC1",
    "rlds_PC1",
    "rsds_PC1",
] 


In [6]:
dict_load_sst_fields = load_sst_fields(src_model, tgt_model, year_end)

sst_src = dict_load_sst_fields["sst_src"]
sst_src_mc = dict_load_sst_fields["sst_src_mc"]
sst_src_histmc = dict_load_sst_fields["sst_src_histmc"]

sst_tgt = dict_load_sst_fields["sst_tgt"]
sst_tgt_mc = dict_load_sst_fields["sst_tgt_mc"]
sst_tgt_anom = dict_load_sst_fields["sst_tgt_anom"]
sst_tgt_anom_sm = dict_load_sst_fields["sst_tgt_anom_sm"]


sst_src_sm_yr = sst_src.groupby("time.year").mean(["time",'lon','lat']).isel(year=slice(1,None)).compute()
sst_tgt_sm_yr = sst_tgt.groupby("time.year").mean(["time",'lon','lat']).isel(year=slice(1,None)).compute()



In [ ]:
lat = sst_tgt.lat
lon = sst_tgt.lon
msk = sst_tgt.maskC
minlon = sst_tgt.lon.min().values
minlat = sst_tgt.lat.min().values
maxlon = sst_tgt.lon.max().values
maxlat = sst_tgt.lat.max().values
minlon, maxlon, minlat, maxlat

(array(117.04166412), array(159.625), array(20.03914452), array(53.18893814))

In [ ]:
dict_load_eofs_pcs = load_eofs_pcs(src_model, tgt_model, SSPLIST, varlist_X, data_dir, year_start, nt, TIME, 0)

pcs_X_src = dict_load_eofs_pcs['pcs_X_src']
pcs_X_tgt = dict_load_eofs_pcs['pcs_X_tgt']
pcs_Y_src = dict_load_eofs_pcs['pcs_Y_src']  #! (time, mode)
eofs_Y_src = dict_load_eofs_pcs['eofs_Y_src']  #! (mode, lat, lon)
evrs_Y_src = dict_load_eofs_pcs['evrs_Y_src']
wgts_Y_src = dict_load_eofs_pcs['wgts_Y_src']

eofs_Y_src = eofs_Y_src.where(sst_src.maskC).persist()

eofs_Y_src_245 = eofs_Y_src.sel(ssp=245)

pcs_Y_src


 -----------Executing function: load_XandY----------

 -----------Executing function: load_XandY----------

 -----------Executing function: load_Y_dataset----------

 -----------Executing function: load_XandY----------

 -----------Executing function: load_XandY----------

 -----------Executing function: load_Y_dataset----------

 -----------Executing function: load_XandY----------

 -----------Executing function: load_XandY----------

 -----------Executing function: load_Y_dataset----------


<xarray.DataArray (ssp: 3, time: 1272, mode: 1272)> Size: 19MB
array([[[ -1.6318043 ,  -1.5564271 ,  -0.1372049 , ...,  -0.        ,
          -0.        ,  -8.142157  ],
        [ -1.692695  ,  -1.9675258 ,  -0.08343799, ...,  -1.1429477 ,
          -2.8149242 ,  -9.763451  ],
        [ -1.6262927 ,  -2.1500762 ,   0.12036781, ...,  -1.7005037 ,
           0.55101776,  -6.4545336 ],
        ...,
        [  0.79827523,   0.24186212,   1.4709889 , ...,  -0.38909352,
           0.26945344,  -8.055243  ],
        [  0.7777216 ,  -0.23864618,   0.17519759, ...,   0.12814116,
           1.1642734 ,  -8.278499  ],
        [  0.5020434 ,  -0.20190053,  -1.0349405 , ...,   0.21965595,
           1.1381258 , -10.648953  ]],

       [[ -1.5057876 ,  -1.4290754 ,   0.72118235, ...,  -0.11696416,
          -0.13077693,   5.448632  ],
        [ -1.5646871 ,  -1.827661  ,   0.8383427 , ...,   1.4214644 ,
          -1.5792601 ,   6.0485716 ],
        [ -1.510983  ,  -2.0827312 ,   0.9489046 , ...,   1.4956539 ,
          -1.4250548 ,   4.655059  ],
...
          -0.06391795,   5.4529486 ],
        [  1.2526066 ,   0.27552187,   0.43607202, ...,   0.6585392 ,
           0.6755667 ,   5.799141  ],
        [  1.4328458 ,   0.24318752,   0.8674128 , ...,  -2.4756355 ,
          -1.2488449 ,   4.635576  ]],

       [[ -1.2389283 ,  -1.7324436 ,  -0.3226454 , ...,   0.8569354 ,
           0.        ,   0.        ],
        [ -1.2639431 ,  -2.1728299 ,  -0.2320645 , ...,   3.2832966 ,
          -1.0223355 ,   1.0926747 ],
        [ -1.2538159 ,  -2.3875628 ,  -0.11489416, ...,  -0.91370577,
          -0.07961065,   3.2609103 ],
        ...,
        [  2.0123472 ,   0.26199988,  -0.15621787, ...,   0.48783112,
           0.3812886 ,   1.0688477 ],
        [  1.7669489 ,  -0.04562766,  -0.40580377, ...,  -0.12974367,
           0.15425575,   1.3819126 ],
        [  1.3825716 ,   0.12860517,  -0.5662738 , ...,   2.0678189 ,
          -0.48818824,  -1.1131607 ]]],
      shape=(3, 1272, 1272), dtype=float32)
Coordinates:
  * time     (time) datetime64[ns] 10kB 1995-01-15 1995-02-15 ... 2100-12-15
  * mode     (mode) int64 10kB 1 2 3 4 5 6 7 ... 1267 1268 1269 1270 1271 1272
  * ssp      (ssp) int64 24B 126 245 585

In [ ]:
pred_Y_tgt = predict_target_pcs(SSPLIST,  ML_dir, pclen_loop, pcs_X_tgt, seq_length, var_trendX, TIME_seq, ssptransfer=ssptransfer,src_model=src_model, tfmodel=tfmodel)

InPc10_Train_KACE-1-0-G_6xvar_SST_ssp126_transformer
/home/dhkim//research/Machine_Learning/MLresults//InPc10_Train_KACE-1-0-G_6xvar_SST_ssp126_transformer
SSP: 126 | Mode: 0

 -----------Executing function: preprocessing_trend----------
(1269, 4, 20)


TypeError: <class 'keras.src.models.functional.Functional'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()` are explicitly deserialized in the model's `from_config()` method.

config={'module': 'keras.src.models.functional', 'class_name': 'Functional', 'config': {}, 'registered_name': 'Functional', 'build_config': {'input_shape': None}, 'compile_config': {'optimizer': {'module': 'keras.optimizers', 'class_name': 'Adam', 'config': {'name': 'adam', 'learning_rate': 0.0001250000059371814, 'weight_decay': None, 'clipnorm': None, 'global_clipnorm': None, 'clipvalue': None, 'use_ema': False, 'ema_momentum': 0.99, 'ema_overwrite_frequency': None, 'loss_scale_factor': None, 'gradient_accumulation_steps': None, 'beta_1': 0.9, 'beta_2': 0.999, 'epsilon': 1e-07, 'amsgrad': False}, 'registered_name': None}, 'loss': 'mse', 'loss_weights': None, 'metrics': None, 'weighted_metrics': None, 'run_eagerly

# Reconstruction

In [ ]:
cedl_8m_anom = reconstructiont_eofs_cedl(pred_Y_tgt, eofs_Y_src, wgts_Y_src, selmode1=1,selmode2=selmodes, ssp_list=SSPLIST)

cedl_8m = (cedl_8m_anom.groupby("time.month") + sst_src_mc).persist()

bias = cedl_8m.sel(time=slice('1995','2014')).groupby('time.month').mean(['time']) - sst_src_histmc
cedl_8m_adjust = cedl_8m.groupby('time.month') - bias

cedl_8m_sm_yr = cedl_8m_adjust.groupby("time.year").mean(dim="time").mean(["lon", "lat"]).isel(year=slice(1, None)).compute()  
cedl_8m_sm_yr_bias = cedl_8m.groupby("time.year").mean(dim="time").mean(["lon", "lat"]).isel(year=slice(1, None)).compute()  


sst_src_hpc = reconstructiont_eofs_cedl(pcs_Y_src, eofs_Y_src, wgts_Y_src, selmode1=selmodes + 1, selmode2=None, ssp_list=SSPLIST).isel(time=slice(seqnt, None))

cedl_full_anom = (cedl_8m_anom + sst_src_hpc).persist()
cedl_full_anom_sm = cedl_full_anom.mean(["lon", "lat"]).compute()

cedl_full = (cedl_8m_adjust + sst_src_hpc).persist()
cedl_full_sm_yr = cedl_full.groupby("time.year").mean(dim="time").mean(["lon", "lat"]).isel(year=slice(1, None)).compute()


NameError: name 'pred_Y_tgt' is not defined

# 기본 시계열 그래프

In [ ]:
lw_true, lw_pred = 1, .8

ssp_colors = ["#DAA520", "#FF4D4D", "#5DD3B6"]

fig, axes = plt.subplots(3, 1, figsize=(5, 4), dpi=200, gridspec_kw={"hspace": 0.0})

axes[0].annotate(f"{tgt_model} SSTA", xy=(0.5, 1.2), xycoords="axes fraction", ha="center", va="bottom", fontsize=10, fontweight="bold")

for i, ax in enumerate(axes):
    ssp = SSPLIST[i]
    truedata = sst_tgt_anom_sm.sel(ssp=ssp)
    preddata = cedl_full_anom_sm.sel(ssp=ssp)
    lrdata = sst_rawtgt_adjust_anom_sm.sel(ssp=ssp)

    ax.plot(TIME[:len(truedata) ], truedata[:], label="DD-SSTA", lw=lw_true, alpha=0.6, color="k", ls="-")
    ax.plot(TIME[seqnt:len(preddata) + seqnt ], preddata[:], label="DL-SSTA", lw=lw_pred, alpha=1, color=ssp_colors[i], ls="-")

    cor = np.corrcoef(truedata[seqnt:], preddata)[0, 1]
    ssp_str = str(ssp)
    ax.set_title(rf"$\mathbf{{{titleabc[i]}}}\;$ SSP{ssp_str[0]}-{ssp_str[1]}.{ssp_str[2]} (r={cor:.2f})", fontsize=8)

    ax.grid(False)
    ax.spines[["top", "right", "bottom", "left"]].set_visible(False)
    ax.axhline(0, color="gray", zorder=0, lw=1, ls="--", alpha=0.5)
    ax.tick_params(axis="both", which="both", left=False, right=False, labelleft=False, labelright=True, labelsize=6)
    ax.grid(True, which='major', axis='y', alpha=0.3, linestyle='-', linewidth=0.25)

    ax.spines["right"].set_visible(True)
    ax.spines["right"].set_alpha(0.3)

    if i == len(axes) - 1:
        ax.spines["bottom"].set_visible(True)
        ax.spines["bottom"].set_alpha(0.3)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        ax.tick_params(axis="x", which="major", direction='in', bottom=True, labelbottom=True, length=2)
    else:
        ax.tick_params(axis="x", which="both", bottom=False, labelbottom=False)

# 커스텀 범례 설정 (첫 번째 서브플롯에 위치)
line_true = mlines.Line2D([], [], lw=lw_true, alpha=0.6, color="k", ls="-")
line_ssp1 = mlines.Line2D([], [], color="#DAA520", lw=lw_pred)
line_ssp2 = mlines.Line2D([], [], color="#FF4D4D", lw=lw_pred)
line_ssp5 = mlines.Line2D([], [], color="#5DD3B6", lw=lw_pred)

axes[0].legend([line_true, (line_ssp1, line_ssp2, line_ssp5)], ["DD", "CEDL"], 
               handler_map={tuple: HandlerTuple(ndivide=None, pad=0.2)}, frameon=False, loc="upper left", ncol=2, fontsize=7)


# Ensemble plot

In [ ]:


ens_list = [
    # "KACE-1-0-G",    #####
    # "EC-Earth3",
    "ACCESS-CM2",     
    "AWI-CM-1-1-MR",  
    "BCC-CSM2-MR",    
    "CMCC-CM2-SR5",   
    "CanESM5-1",      
    "FGOALS-f3-L",    
    "GFDL-ESM4", 
    "IPSL-CM6A-LR", 
    "KIOST-ESM", 
    "MIROC6",         
    "MPI-ESM1-2-LR",  
    "MRI-ESM2-0",     
    "NorESM2-LM",
]
ens_list = [tgt_model]+ens_list
ens_list

In [ ]:

pred_Y_ens = {}
for model in ens_list:
    pcs_X_ens={}
    print(f"Predicting for target model: {model}")
    for ssp in SSPLIST:
        pcs_X_ens[ssp] = load_X_dataset(
            model=model,
            SSP=ssp,
            varlist=varlist_X,
            set_input=set_input,
            useEOF85=0,
            data_dir=data_dir,
            year_start=year_start,
        )

    pred_Y_ens[model] = predict_target_pcs(SSPLIST,  ML_dir, pclen_loop, pcs_X_ens, seq_length, var_trendX, TIME_seq, ssptransfer=ssptransfer,src_model=src_model, tfmodel=tfmodel,set_input=set_input)

In [ ]:

cedl_anom_ens = {}
cedl_full_ens = {}
cedl_full_ens_adjust = {}
cedl_full_ens_sm_yr = {}
cedl_full_ens_adjust_sm_yr={}
for model in ens_list:
    print(f"Reconstructing for target model: {model}")
    cedl_anom_ens[model] = reconstructiont_eofs_cedl(pred_Y_ens[model], eofs_Y_src, wgts_Y_src, selmode1=1,selmode2=selmodes, ssp_list=SSPLIST)
        
    tmp_season = cedl_anom_ens[model].groupby('time.month') + sst_src_mc    
    tmp_seasen_full = tmp_season+sst_src_hpc #! higher-order add
    
    bias = tmp_seasen_full.sel(time=slice('1995-04','2014-12')).groupby('time.month').mean('time') - sst_src_histmc
    cedl_full_ens_adjust[model] = tmp_seasen_full.groupby('time.month') - bias
    
    cedl_full_ens_sm_yr[model] = tmp_seasen_full.groupby('time.year').mean(['time','lon','lat']).isel(year=slice(1,None)).compute()
    cedl_full_ens_adjust_sm_yr[model] = cedl_full_ens_adjust[model].groupby('time.year').mean(['time','lon','lat']).isel(year=slice(1,None)).compute()
    
cedl_full_ens_sm_yr= xr.concat(
    list(cedl_full_ens_sm_yr.values()), 
    dim='model'
).assign_coords(model=list(cedl_full_ens_sm_yr.keys()))

cedl_full_ens_adjust_sm_yr= xr.concat(
    list(cedl_full_ens_adjust_sm_yr.values()), 
    dim='model'
).assign_coords(model=list(cedl_full_ens_adjust_sm_yr.keys()))



In [ ]:
sst_rawtgt_histmc_1d = sst_rawtgt_histmc.mean(['month','lat','lon'])
sst_rawtgt_histmc_1d

In [ ]:
fig = plt.figure(figsize=(7.09, 1.50), dpi=200)
outer_gs = gridspec.GridSpec(1, 3, wspace=0.3) 

ts_axes, box_axes = [], []

for i in range(3):
    inner_gs = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=outer_gs[i], width_ratios=[5, 1], wspace=0.0)
    
    if i == 0:
        ax = fig.add_subplot(inner_gs[0])
        box_ax = fig.add_subplot(inner_gs[1], sharey=ax)
        first_ax = ax
    else:
        ax = fig.add_subplot(inner_gs[0], sharey=first_ax)
        box_ax = fig.add_subplot(inner_gs[1], sharey=first_ax)
        ax.tick_params(labelleft=False)
        
    ts_axes.append(ax)
    box_axes.append(box_ax)

for i, ax in enumerate(ts_axes):
    ssp, box_ax = SSPLIST[i], box_axes[i]
    da = cedl_full_ens_adjust_sm_yr.sel(ssp=ssp)
    
    ax.plot(sst_src_sm_yr.year, sst_src_sm_yr.sel(ssp=ssp), label=f"{src_model}, DD", lw=1, alpha=0.8, color="b", ls="-", zorder=1)
    ax.plot(sst_tgt_sm_yr.year, sst_tgt_sm_yr.sel(ssp=ssp), label=f"{tgt_model}, DD", lw=1, alpha=0.8, color="r", ls="-", zorder=1)

    for model in ens_list:
        ax.plot(da.year, da.sel(model=model), color="gray", zorder=0.1, alpha=0.4, lw=0.5)
        if model == tgt_model:
            ax.plot(da.year, da.sel(model=model), color="k", zorder=2, lw=1, alpha=0.99, ls=":", label="EC SST, CEDL")
    ax.plot(da.year, da.mean("model"), label="Ens. mean", lw=2, alpha=0.7, color="k", ls="-", zorder=0.8)

    fillmax, fillmin = da.quantile(0.975, dim="model"), da.quantile(0.025, dim="model")
    ax.fill_between(da.year, fillmin, fillmax, color="lightgray", alpha=0.8, label="95% spread", zorder=0.0) 
    da_2100_1 = da.sel(year=2100).values
    da_2100_2 = ds_result.sel(year=2100, ssp=ssp).values
        
    ax.set_xlim(1995, 2100)
    